In [1]:
from pyspark.sql import SparkSession

WAREHOUSE = "s3://metacode-iceberg-ming/warehouse/"

spark = (
    SparkSession.builder.appName("IcebergGlueLab")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.glue_catalog.warehouse", WAREHOUSE)
    .getOrCreate()
)

print("Spark 세션 생성 완료!")

Spark 세션 생성 완료!


In [3]:
#Glue에 ad_lakehouse 데이터베이스가 없어서 생긴 오류야. 테이블 만들기 전에 데이터베이스를 먼저 만들어야 해.
spark.sql("CREATE DATABASE IF NOT EXISTS glue_catalog.ad_lakehouse")
print("데이터베이스 생성 완료!")

데이터베이스 생성 완료!


In [4]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS glue_catalog.ad_lakehouse.events (
        event_id    BIGINT,
        event_time  TIMESTAMP,
        user_id     STRING,
        amount      DOUBLE
    )
    PARTITIONED BY (day(event_time))
    LOCATION 's3://metacode-iceberg-ming/warehouse/ad_lakehouse.db/events/'
    TBLPROPERTIES (
        'table_type' = 'ICEBERG',
        'format'     = 'parquet',
        'format_version' = '2'
    )
""")
print("테이블 생성 완료!")

테이블 생성 완료!


In [6]:
spark.sql("""
    INSERT INTO glue_catalog.ad_lakehouse.events VALUES
    (1, TIMESTAMP '2026-04-10 09:00:00', 'u1', 100.0),
    (2, TIMESTAMP '2026-04-10 10:00:00', 'u2', 200.0)
""")
print("INSERT 1번 완료!")

INSERT 1번 완료!


In [7]:
spark.sql("""
    INSERT INTO glue_catalog.ad_lakehouse.events VALUES
    (3, TIMESTAMP '2026-04-11 11:00:00', 'u1', 150.0)
""")
print("INSERT 2번 완료!")

INSERT 2번 완료!


In [9]:
spark.sql("""
    SELECT snapshot_id, parent_id, operation
    FROM glue_catalog.ad_lakehouse.events.snapshots
""").show()

+-------------------+-------------------+---------+
|        snapshot_id|          parent_id|operation|
+-------------------+-------------------+---------+
|6678246332117817747|               NULL|   append|
|3373114001240471164|6678246332117817747|   append|
|8577338867622345754|3373114001240471164|   append|
+-------------------+-------------------+---------+



In [10]:
spark.sql("""
    SELECT file_path, record_count, lower_bounds, upper_bounds
    FROM glue_catalog.ad_lakehouse.events.files
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------+
|file_path                                                                                                                                              |record_count|lower_bounds                                                                                                  |upper_bounds                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------------